# Write Sampled Events to a Custom NeXus File

We generate a real McStas NeXus file and stream-sample its weighted events into separate **pixel IDs** and **time offsets from the last pulse**. The first example starts with `create_empty_template()` and builds a minimal output file.

The second example uses the McStas file as both reference template and source. It retains the instrument, geometry, simulation parameters, and other metadata, replacing only the selected detector's raw event data. Both examples use the same shared sampling specification; the original file stays untouched.

Running this notebook requires McStas and McStasScript, as in the other simulation demos, plus NumPy and h5py. Scipp is not required. All generated instrument and data files go into a temporary working directory.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import tempfile

import h5py
import mcstasscript as ms
import numpy as np

import mcstastox as mx

## Generate a Small McStas Source File

The instrument is deliberately simple: `Source_simple` illuminates a 4 by 4 pixel detector two metres away. `Monitor_nD` records neutron weight `p`, detector pixel ID `id`, and arrival time `t`, among its other columns. We will select only this detector.

To use your own simulation later, replace this cell with `source_path = Path("path/to/mccode.h5")` and choose the appropriate component in the sampling cell.

In [2]:
work_dir = Path(tempfile.mkdtemp(prefix="mcstastox_custom_nexus_"))
instrument = ms.McStas_instr("custom_nexus_demo", input_path=str(work_dir))

source = instrument.add_component("source", "Source_simple")
source.set_parameters(
    xwidth=0.01, yheight=0.01,
    focus_xw=0.1, focus_yh=0.1, dist=2.0,
    lambda0=instrument.add_parameter("wavelength", value=1.8),
    dlambda=instrument.add_parameter("delta_wavelength", value=0.5),
)
detector = instrument.add_component("Detector", "Monitor_nD")
detector.set_parameters(
    xwidth=0.1, yheight=0.1, filename='"detector_events.dat"',
    options='"mantid square x bins=4 y bins=4, neutron pixel min=0 t, list all neutrons"',
)
detector.set_AT(2.0, RELATIVE=source)
instrument.settings(
    ncount=10_000, seed=17, NeXus=True,
    output_path=str(work_dir / "simulation"), suppress_output=True,
)
_ = instrument.backengine()
source_path = work_dir / "simulation" / "mccode.h5"
assert source_path.is_file()
print(f"McStas source: {source_path}")

McStas source: /var/folders/1t/8gxgl7l51dd3pk_tbw0mvtj80000gn/T/mcstastox_custom_nexus_9ploegvr/simulation/mccode.h5


## Define the Pulse Clock Explicitly

McStas `t` is the particle arrival time in seconds, not automatically a time offset from the latest source pulse. For an ideal periodic clock with pulse origin $t_0$ and period $T$, we define

$$t_{\mathrm{offset}} = (t - t_0) \bmod T.$$

Here we choose **50 Hz**, a **20 ms period**, and **origin zero** as an explicit analysis convention. `Source_simple` emits at `t=0`; this small simulation does not itself generate a pulse train. Its flight times are shorter than this period, so its offsets normally equal the arrival times. The same transformation wraps later arrivals, as the small boundary check below demonstrates. For real data, supply the clock and time origin that match the simulation or experiment.

This is a two-column event example, not a complete facility-specific pulse layout. It does not preserve which pulse an event belongs to or construct `event_index`/`event_time_zero` arrays.

In [3]:
pulse_frequency_hz = 50.0
pulse_period_s = 1.0 / pulse_frequency_hz
pulse_origin_s = 0.0


def time_since_last_pulse(arrival_time_s):
    return np.remainder(arrival_time_s - pulse_origin_s, pulse_period_s)


probe_times = pulse_origin_s + pulse_period_s * np.array([0.25, 1.25, 2.25])
np.testing.assert_allclose(
    time_since_last_pulse(probe_times), pulse_period_s * np.full(3, 0.25)
)

## Describe One Shared Stream Sample

Select 1,000 unit-weight events with replacement from the weighted McStas stream. The original weight `p` is read automatically even though we request only `id` and `t` as output variables. Repeated events are allowed; output order is not meaningful. Reusing this **same `events` object** keeps the two arrays aligned and invokes sampling once per write.

Input reads are chunked. The existing sampler still retains the requested sample in memory; output batching only bounds the subsequent transformations and writes. A fixed seed reproduces the selection for fixed input and read chunk size.

In [4]:
sampling = mx.SamplingSettings(n_samples=1_000, seed=42)
events = mx.event_source(
    component_name="Detector", variables=("id", "t"), read_chunk_size=256,
).sample(sampling)

pixel_ids = events["id"]
time_offsets_us = events["t"].map(
    time_since_last_pulse, dtype="float64",
    description=(
        f"Time since last {pulse_frequency_hz:g} Hz pulse; origin {pulse_origin_s:g} s"
    ),
).convert_units("s", "us", dtype="float64")

## Example 1: Start from an Empty Template

Start with `create_empty_template()` and describe a small layout from scratch. `/entry1/events/event_id` contains integer McStas pixel IDs, unchanged in numbering. `/entry1/events/event_time_offset` contains floating-point offsets in **microseconds**. The pulse convention is stored alongside the events.

Each row represents one normal event, so no weight dataset is needed. `duration` comes from the shared sampling result: when McStas weights represent count rates, it is `n_samples / total_weight` seconds. No timestamp or acquisition duration is inherited from the reference.

In [5]:
def add_event_layout(layout):
    layout.attrs["NX_class"] = "NXroot"
    layout.attrs["creator"] = "McStasToX custom NeXus demo"
    entry = layout.add_group("/entry1")
    entry.attrs["NX_class"] = "NXentry"
    entry.add_dataset("title", value="Sampled McStas detector events")
    entry.add_dataset("source_file", value=str(source_path))
    entry.add_dataset("pulse_period", value=pulse_period_s).attrs["units"] = "s"
    entry.add_dataset("pulse_origin", value=pulse_origin_s).attrs["units"] = "s"
    entry.add_dataset("duration").set_from_result(events, "effective_duration")
    entry["duration"].attrs["units"] = "s"

    output_events = entry.add_group("events")
    output_events.attrs["NX_class"] = "NXevent_data"
    output_events.add_dataset("event_id", shape=(0,), dtype="int64").copy_from(pixel_ids)
    output_events.add_dataset(
        "event_time_offset", shape=(0,), dtype="float64",
    ).copy_from(time_offsets_us)
    output_events["event_time_offset"].attrs["units"] = "us"
    return layout


template = add_event_layout(mx.create_empty_template())
template.validate().raise_if_errors()
print(template.summary())

NexusTemplate: base=empty
sample1: weighted sample requested=1000, seed=42; events from Detector; variables=(id, t); read_chunk_size=256; filter_zeros=False; weighted sample requested=1000, seed=42, ordered=False, weight_key='...
Nodes: 10 (attributes do not imply data transformations)
/: group; new/replaced; 
  @NX_class: literal NXroot
  @creator: literal McStasToX custom NeXus demo
/entry1: group; new/replaced; 
  @NX_class: literal NXentry
/entry1/duration: dataset; new/replaced; 
  sample1 result -> effective_duration
  @units: literal s
/entry1/events: group; new/replaced; 
  @NX_class: literal NXevent_data
/entry1/events/event_id: dataset; new/replaced; shape=[0]; dtype=int64
  sample1 -> id
/entry1/events/event_time_offset: dataset; new/replaced; shape=[0]; dtype=float64
  sample1 -> t -> custom callback time_since_last_pulse (Time since last 50 Hz pulse; origin 0 s); dtype=float64 -> convert s to us (multiply by 1e+06); dtype=float64
  @units: literal us
/entry1/pulse_origin: 

### Write the Minimal File

The empty template has no reference file; `source_path` is used only to read the McStas events. The writer samples once and writes aligned output batches into a **different file**. `overwrite=True` allows rerunning this cell against its own demo output, never overwriting the source.

The summary can describe the standard unit conversion and shows our description for the custom pulse-folding callback. It does not infer a mathematical formula from Python code.

In [6]:
output_path = work_dir / "sampled_events_from_empty.nxs"
report = mx.write_nexus(
    template, output_path, source=source_path, write_chunk_size=200, overwrite=True,
)
assert report.sources[events].n_selected == sampling.n_samples
assert template.base_path is None
assert template.retained_datasets() == []
print(report)
print(f"Output: {output_path}")

WriteReport(output='/var/folders/1t/8gxgl7l51dd3pk_tbw0mvtj80000gn/T/mcstastox_custom_nexus_9ploegvr/sampled_events_from_empty.nxs', sources=1, datasets=7, diagnostics=0)
Output: /var/folders/1t/8gxgl7l51dd3pk_tbw0mvtj80000gn/T/mcstastox_custom_nexus_9ploegvr/sampled_events_from_empty.nxs


### Read Back the Two Arrays

For this small demonstration we load the 1,000 output rows to inspect them. A larger output can be inspected with slices instead. Check the length, dtypes, units, and offset range.

In [7]:
with h5py.File(output_path, "r") as written:
    output_ids = written["/entry1/events/event_id"][:]
    output_offsets_us = written["/entry1/events/event_time_offset"][:]
    assert set(written) == {"entry1"}
    assert output_ids.dtype == np.dtype("int64")
    assert output_offsets_us.dtype == np.dtype("float64")
    assert written["/entry1/events/event_time_offset"].attrs["units"] == "us"
    assert written["/entry1/duration"][()] == report.sources[events].effective_duration

assert output_ids.shape == output_offsets_us.shape == (sampling.n_samples,)
assert np.all((0 <= output_ids) & (output_ids < 16))
assert np.all(np.isfinite(output_offsets_us))
assert np.all((0 <= output_offsets_us) & (output_offsets_us < pulse_period_s * 1e6))
[(int(pixel), float(offset)) for pixel, offset in zip(
    output_ids[:5], output_offsets_us[:5], strict=True
)]

[(14, 1112.5084761163614),
 (15, 802.4255880750143),
 (10, 756.9732384516972),
 (4, 1049.2398689437919),
 (8, 732.1330251880806)]

As a correctness check, repeat the small seeded selection using the same input chunks, then compare **both columns** with the output. This additional sampling call is only for verification; the conversion itself needed one call inside `write_nexus`.

In [8]:
with mx.SourceContext(source_path) as original:
    expected = mx.sample_event_chunks(events.iter_chunks(original), sampling)
np.testing.assert_array_equal(output_ids, expected["id"].astype("int64"))
np.testing.assert_array_equal(
    output_offsets_us, time_since_last_pulse(expected["t"]) * 1e6
)
print("Pixel IDs and pulse-relative times refer to the same sampled events.")

Pixel IDs and pulse-relative times refer to the same sampled events.


## Example 2: Retain the McStas Metadata

Now use `create_template(source_path)` without clearing the reference. Keep its instrument description, component positions/rotations, detector geometry, pixel definitions, simulation parameters, and other metadata. Replace only the selected detector's raw weighted `events` table with the two sampled arrays.

McStas exposes the same event **group** through two hard-link paths: one below `/entry1/data` and another below the detector's `instrument/components/.../output`. Discover those paths through the reader rather than hard-coding the generated group name. Editing a child of that shared group affects both views automatically. Deleting only one group link would leave the old events reachable through the other.

For a source with more detectors, this example changes only `Detector`; other monitors would remain unchanged.

In [9]:
with mx.SourceContext(source_path) as context:
    detector_path = context.mcstas.get_component_entry("Detector").name
    info = context.mcstas.get_info_entry("Detector")
    event_group_path = info.name
    data_alias_path = next(
        obj.name for obj in context.raw["/entry1/data"].values()
        if isinstance(obj, h5py.Group) and obj.id == info.id
    )
    original_event_shape = info["events"].shape
    original_event_attributes = dict(info.attrs)
    original_root_attributes = dict(context.raw.attrs)

reference_template = mx.create_template(source_path)
original_retained = reference_template.retained_datasets()
reference_template.validate().raise_if_errors()
print(f"Shared group: {event_group_path}\nData alias: {data_alias_path}")
print(f"Original event table: {original_event_shape}")
print(f"Retained datasets before replacement: {len(original_retained)}")

Shared group: /entry1/instrument/components/0001_Detector/output/detector_events_dat_list_p_x_y_n_id_t
Data alias: /entry1/data/detector_events_dat_list_p_x_y_n_id_t
Original event table: (10000, 6)
Retained datasets before replacement: 21


### Replace Data, Not the Instrument

Delete the shared group's `events` child, not the group itself, then add `event_id` and `event_time_offset` through that same group. We also add `/entry1/events` as a convenient extra hard link; the original two paths remain valid.

Descriptions such as `variables`, `signal`, and statistics refer to the old weighted table. Archive those attributes as `mcstas_original_*` provenance instead of presenting them as descriptions of the new arrays. Keep component identity and other metadata in place. Store the new pulse convention and sampled exposure separately under `/entry1/sampling`, leaving the original simulation duration/timestamps intact.

Root file identity attributes are updated for the converted file while their original values are also preserved. The detector's existing `target` path remains valid because we do not move its group.

In [11]:
output_events = reference_template[event_group_path]
del output_events["events"]
old_table_attributes = (
    "NX_class", "variables", "dataset columns", "signal", "statistics",
    "values", "type", "xylimits", "xlabel", "ylabel", "zlabel",
    "xvar", "yvar", "zvar", "title",
)
for name in old_table_attributes:
    if name in original_event_attributes:
        output_events.attrs[f"mcstas_original_{name}"] = original_event_attributes[name]
        del output_events.attrs[name]
output_events.attrs["NX_class"] = "NXevent_data"
output_events.attrs["variables"] = "event_id event_time_offset"
output_events.attrs["dataset columns"] = "event_id event_time_offset"
output_events.attrs["title"] = "Sampled unit-weight detector events"
output_events.add_dataset("event_id", shape=(0,), dtype="int64").copy_from(pixel_ids)
output_events.add_dataset(
    "event_time_offset", shape=(0,), dtype="float64",
).copy_from(time_offsets_us)
output_events["event_time_offset"].attrs["units"] = "us"
reference_template.create_link("/entry1/events", event_group_path)

sampling_metadata = reference_template.add_group("/entry1/sampling")
sampling_metadata.attrs["NX_class"] = "NXcollection"
sampling_metadata.add_dataset("pulse_period", value=pulse_period_s).attrs["units"] = "s"
sampling_metadata.add_dataset("pulse_origin", value=pulse_origin_s).attrs["units"] = "s"
sampling_metadata.add_dataset("effective_duration").set_from_result(
    events, "effective_duration"
)
sampling_metadata["effective_duration"].attrs["units"] = "s"

reference_output_path = work_dir / "sampled_events_with_metadata.nxs"
for name in ("file_name", "file_time", "creator"):
    if name in original_root_attributes:
        reference_template.attrs[f"mcstas_original_{name}"] = original_root_attributes[name]
reference_template.attrs["file_name"] = str(reference_output_path)
reference_template.attrs["file_time"] = datetime.now(timezone.utc).isoformat()
reference_template.attrs["creator"] = f"McStasToX {mx.__version__}"

retained = reference_template.retained_datasets()
assert len(retained) == len(original_retained) - 1
assert not any(record.path.endswith("/events") for record in retained)
reference_template.validate().raise_if_errors()
print(f"Keeping {len(retained)} original metadata datasets; removed one raw event table.")
print(reference_template.summary(limit=8))

Keeping 20 original metadata datasets; removed one raw event table.
NexusTemplate: base=/var/folders/1t/8gxgl7l51dd3pk_tbw0mvtj80000gn/T/mcstastox_custom_nexus_9ploegvr/simulation/mccode.h5
sample1: weighted sample requested=1000, seed=42; events from Detector; variables=(id, t); read_chunk_size=256; filter_zeros=False; weighted sample requested=1000, seed=42, ordered=False, weight_key='...
Nodes: 85 (attributes do not imply data transformations)
/: group; inherited from /; 
  @HDF5_Version: literal b'1.14.4'
  @NeXus_version: literal b'4.4.2'
  @creator: literal McStasToX 0.0.13.dev4+gad08a2bf5
  ... 5 more attributes
/entry1: group; inherited from /entry1; 
  @NX_class: literal b'NXentry'
  @instrument: literal b'custom_nexus_demo'
  @program_name: literal b' 3.6.16, git'
  ... 1 more attributes
/entry1/data: group; inherited from /entry1/data; 
  @NX_class: literal b'NXdetector'
/entry1/data/detector_events_dat_list_p_x_y_n_id_t: group; inherited from /entry1/data/detector_events_da

In [12]:
reference_report = mx.write_nexus(
    reference_template, reference_output_path, source=source_path,
    write_chunk_size=200, overwrite=True,
)
assert reference_report.sources[events].n_selected == sampling.n_samples
print(reference_report)
print(f"Metadata-preserving output: {reference_output_path}")

WriteReport(output='/var/folders/1t/8gxgl7l51dd3pk_tbw0mvtj80000gn/T/mcstastox_custom_nexus_9ploegvr/sampled_events_with_metadata.nxs', sources=1, datasets=29, diagnostics=3)
Metadata-preserving output: /var/folders/1t/8gxgl7l51dd3pk_tbw0mvtj80000gn/T/mcstastox_custom_nexus_9ploegvr/sampled_events_with_metadata.nxs


### Verify What Was Retained and Replaced

This second write performs a fresh sample. The fixed seed and unchanged input read partition reproduce the first example's event arrays. Verify those arrays through **all three group paths**, confirm that the old raw table is absent, and compare the small geometry/parameter datasets with the original. The original file must still contain its full weighted table.

Hard-link warnings in the template summary are expected: they explain that edits to the shared event group affect every alias. They do not mean the geometry or simulation metadata was discarded.

In [13]:
def assert_metadata_equal(original, copied):
    assert set(original.attrs) == set(copied.attrs)
    for name in original.attrs:
        np.testing.assert_array_equal(original.attrs[name], copied.attrs[name])
    if isinstance(original, h5py.Group):
        assert set(original) == set(copied)
        for name in original:
            assert_metadata_equal(original[name], copied[name])
    else:
        assert original.dtype == copied.dtype
        np.testing.assert_array_equal(original[()], copied[()])


metadata_paths = [
    "/entry1/simulation",
    f"{detector_path}/Geometry", f"{detector_path}/Position",
    f"{detector_path}/Rotation", f"{detector_path}/parameters",
    f"{detector_path}/output/BINS",
]
with h5py.File(source_path, "r") as original, h5py.File(reference_output_path, "r") as written:
    for path in (event_group_path, data_alias_path, "/entry1/events"):
        assert written[path].id == written[event_group_path].id
        assert "events" not in written[path]
        np.testing.assert_array_equal(written[path + "/event_id"][:], output_ids)
        np.testing.assert_array_equal(
            written[path + "/event_time_offset"][:], output_offsets_us
        )
    for path in metadata_paths:
        assert_metadata_equal(original[path], written[path])
    assert original[event_group_path + "/events"].shape == original_event_shape
    np.testing.assert_array_equal(
        written[event_group_path].attrs["mcstas_original_variables"],
        original_event_attributes["variables"],
    )
    assert written["/entry1/sampling/effective_duration"][()] == report.sources[events].effective_duration

print("Geometry, pixel definitions, component metadata, and simulation parameters are retained.")
print("All event-group aliases expose the same sampled IDs and pulse-relative times.")

Geometry, pixel definitions, component metadata, and simulation parameters are retained.
All event-group aliases expose the same sampled IDs and pulse-relative times.


The first output contains only the minimal layout we built from scratch. The second retains the McStas metadata and geometry, but replaces the detector's weighted event table with the same sampled IDs and time offsets. Its original simulation duration and timestamps remain provenance; the new sampled exposure is explicitly separate. Both outputs remain in `work_dir` for inspection, and `source_path` is unchanged.

The metadata-rich output is a custom format: retaining a McStas instrument tree does not restore the raw `events` table expected by the McStas event reader. For a facility format, keep the required reference metadata, supply an ID mapping when detector numbering differs, and implement the required pulse grouping and time references. See [Custom NeXus Files](custom_nexus.md) for supported operations and memory limits.